# Shared prior samples across forward models

Montage of four realizations of $w \sim N(0, C)$ and the corresponding $(m, u=F(m))$ for Poisson, linear elasticity, and hyperelasticity. All three problems share the same prior samples $w$; each model applies its own $(\alpha_m, \beta_m)$ map and forward operator $F$.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = os.getcwd()
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", ".."))
sys.path.insert(0, os.path.join(ROOT, "src/plotting"))

from plot_mix_collection import get_default_plot_mix_collection_data, plot_mix_collection

plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
seed = 0
np.random.seed(seed)

N_ROWS = 4
N_COLS = 7

SURVEY_WORK = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
PROBLEM_DATA = {
    "Poisson": os.path.join(SURVEY_WORK, "problems/poisson/data/Poisson_samples.npz"),
    "Linear Elasticity": os.path.join(
        SURVEY_WORK, "problems/linear_elasticity/data/LinearElasticity_samples.npz"
    ),
    "Hyperelasticity": os.path.join(
        SURVEY_WORK, "problems/hyperelasticity/data/Hyperelasticity_samples.npz"
    ),
}

OUTPUT_DIR = NOTEBOOK_DIR
OUTPUT_PNG = os.path.join(OUTPUT_DIR, "all_model_samples.png")

M_FORMULA = r"$m = \alpha_m\, \exp(w) + \beta_m$"
U_FORMULA = r"$u = F(m)$"
W_TITLE = r"$w \sim N(0, C)$"
SUP_TITLE = (
    r"Shared prior samples $w \sim N(0, C)$ and corresponding "
    r"$m$, $u=F(m)$ for Poisson, linear elasticity, and hyperelasticity"
)

In [ ]:
def load_problem_samples(path: str) -> dict:
    data = np.load(path)
    return {
        "w": data["w_samples"],
        "m": data["m_samples"],
        "u": data["u_samples"],
        "nodes": data["m_mesh_nodes"],
        "alpha_m": float(data["prior_alpham"]),
        "beta_m": float(data["prior_betam"]),
    }


samples = {name: load_problem_samples(path) for name, path in PROBLEM_DATA.items()}
nodes = samples["Poisson"]["nodes"]
n_total = samples["Poisson"]["w"].shape[0]

for name, payload in samples.items():
    if not np.allclose(payload["w"], samples["Poisson"]["w"]):
        raise ValueError(f"w samples differ between Poisson and {name}")
    print(
        f"{name}: n={payload['w'].shape[0]}, "
        f"alpha_m={payload['alpha_m']}, beta_m={payload['beta_m']}, "
        f"u_dim={payload['u'].shape[1]}"
    )

In [ ]:
def two_line_title(model_name: str, formula: str) -> str:
    return f"{model_name}\n{formula}"


def build_column_titles() -> list[str | None]:
    """Header row titles for columns 0..6."""
    return [
        W_TITLE,
        two_line_title("Poisson", M_FORMULA),
        two_line_title("Poisson", U_FORMULA),
        two_line_title("Linear Elasticity", M_FORMULA),
        two_line_title("Linear Elasticity", U_FORMULA),
        two_line_title("Hyperelasticity", M_FORMULA),
        two_line_title("Hyperelasticity", U_FORMULA),
    ]


def build_montage(sample_indices: np.ndarray) -> None:
    col_titles = build_column_titles()

    u_vec = []
    title_vec = []
    cmapvec = []
    flag_is_vec = []
    flag_add_disp = []
    plot_type = []

    for row, sample_idx in enumerate(sample_indices):
        row_fields = [
            samples["Poisson"]["w"][sample_idx, :],
            samples["Poisson"]["m"][sample_idx, :],
            samples["Poisson"]["u"][sample_idx, :],
            samples["Linear Elasticity"]["m"][sample_idx, :],
            samples["Linear Elasticity"]["u"][sample_idx, :],
            samples["Hyperelasticity"]["m"][sample_idx, :],
            samples["Hyperelasticity"]["u"][sample_idx, :],
        ]
        u_vec.append(row_fields)

        if row == 0:
            title_vec.append(col_titles)
        else:
            title_vec.append([None] * N_COLS)

        cmapvec.append(["magma", "jet", "jet", "jet", "jet", "jet", "jet"])
        flag_is_vec.append([False, False, False, False, True, False, True])
        flag_add_disp.append([False, False, False, False, True, False, True])
        plot_type.append(["field"] * N_COLS)

    plot_data = get_default_plot_mix_collection_data(
        rows=N_ROWS,
        cols=N_COLS,
        nodes=nodes,
        figsize=(34, 18),
        fs=18,
        sup_title=SUP_TITLE,
        y_sup_title=1.02,
        savefilename=OUTPUT_PNG,
        u=u_vec,
        cmap=cmapvec,
        title=title_vec,
        is_vec=flag_is_vec,
        add_disp=flag_add_disp,
        plot_type=plot_type,
    )
    plot_mix_collection(plot_data)


sample_indices = np.random.choice(n_total, N_ROWS, replace=False)
print("Sample indices:", sample_indices)
build_montage(sample_indices)